
# Trees and Ensembles, Notebook 6 (bonus track)
## C5.0 with `c50py`: gain ratio, categories without dummies, missing values, pessimistic pruning and built-in boosting

**Prepared by:** David Díaz, with assistance from Claude (Anthropic) · **Environment:** Google Colab

### What this is about

At the end of Notebook 2 we said that after ID3 the story split into C4.5 and CART. Everything that came afterwards followed the
CART branch: Gini, binary splits on numbers, bagging, random forest, gradient boosting. This
bonus track walks the other branch, Quinlan's own: **C4.5** (1993) and its commercial version
**C5.0** (also called See5), which for years was the reference tree in industry and which
in R is used through the `C50` package. Quinlan released the C source code of C5.0 in 2011, and
starting from that code David Díaz, the author of this material, wrote **`c50py`** (`pip install c50py`),
the first implementation of C5.0 in Python: from scratch in Python and NumPy, with the
scikit-learn interface (it fits into a `Pipeline` or a `GridSearchCV`), and with things that neither the original C5.0 nor
R's `C50` package, which only wraps that code, offer: a `C5Regressor`, the exact rule
that was applied to each prediction, and the same tree drawings as scikit-learn (`plot_tree`,
`export_graphviz`, `export_text`). This notebook uses version 0.4.0 or later, which brings pessimistic
pruning and split selection exactly as they are in the C4.5 code.

**What matters for the business** are four things C5.0 does that with scikit-learn you have to
solve by hand: it works with numeric and categorical variables at the same time without preprocessing;
when a categorical variable has many values, it groups the values in the split instead of needing a
column of zeros and ones per value; it handles missing values without imputing; and for each customer
it returns the exact rule that fired. With `trials = 1` the result is a smaller, more
readable tree than scikit-learn's with equal or better performance; with `trials = 10` it becomes a
boosted ensemble without changing anything else. We start there, with a business case, and then
open up each mechanism.

| | CART (scikit-learn) | C5.0 (`c50py`) |
|---|---|---|
| Criterion | Gini or variance | **gain ratio** (gain / entropy of the partition) |
| Categories | must be converted into dummies | used directly: splits by **subsets** of values |
| Missing values | must be imputed beforehand | the observation goes down **both branches** with weights |
| Pruning | cost-complexity with validation | **pessimistic**, using training data only, with a confidence factor `cf` |
| Boosting | separate library | built in: `trials = 10` |
| Output | the tree | the tree, the **rules** as text, and the rule applied to each prediction |

### What you will learn today

1. A customer churn case with numeric and categorical variables, gaps, and the rule that fires for each customer: `c50py` against scikit-learn, tree against tree and ensemble against ensemble.
2. Why C5.0 splits categories into two subsets, and what the gain ratio is.
3. What it does with a missing value, when training and when predicting.
4. How it prunes without validation data, with the C4.5 arithmetic done by hand.
5. The built-in boosting and the exported rules.
6. What all of this gives on our 219 firms, compared with CART.


In [ ]:

import numpy as np
import pandas as pd
import warnings
import itertools

import subprocess, sys
try:
    import c50py
    _version_ok = tuple(int(v) for v in c50py.__version__.split(".")[:2]) >= (0, 4)
except ImportError:
    _version_ok = False
if not _version_ok:   # c50py 0.4.0 or later: draws trees with plot_tree, like scikit-learn
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "c50py>=0.4.0"], check=True)
    import c50py
from c50py import C5Classifier, C5Regressor
assert tuple(int(v) for v in c50py.__version__.split(".")[:2]) >= (0, 4), "This notebook needs c50py 0.4.0 or later: pip install --upgrade c50py (and restart the runtime)"
print("c50py", c50py.__version__)
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.metrics import accuracy_score

warnings.filterwarnings("ignore")
pd.set_option("display.width", 120)
print("c50py ready.")



## 1. The business case: customer churn

A telecommunications company with 3,000 customers wants to know who is going to leave, and why,
so it can do something beforehand. For each customer it knows the **region** (16 regions of Chile), the **plan**
(prepaid, postpaid, business), tenure in months, monthly spend, complaints in the
last 12 months, the gigabytes of data used and age. The data are simulated with a
fixed seed (the code is below, so the example is reproducible and can be modified): churn
depends above all on a group of regions with poor coverage, on complaints, on
tenure and on the plan, and 8% of customers have no recorded data usage or age.


In [ ]:

def simular_churn(n=3000, semilla=2026):
    rng = np.random.default_rng(semilla)
    REGIONES = ["Arica y Parinacota", "Tarapacá", "Antofagasta", "Atacama", "Coquimbo", "Valparaíso", "Metropolitana", "O'Higgins",
                "Maule", "Ñuble", "Biobío", "La Araucanía", "Los Ríos", "Los Lagos", "Aysén", "Magallanes"]
    pesos = np.array([1, 1, 2, 1, 2, 5, 12, 2, 2, 1, 4, 2, 1, 2, 0.5, 0.5]); pesos = pesos / pesos.sum()
    region = rng.choice(REGIONES, n, p=pesos)
    plan = rng.choice(["prepago", "postpago", "empresa"], n, p=[0.45, 0.45, 0.10])
    antig = rng.integers(1, 120, n)
    gasto = np.round(np.where(plan == "empresa", rng.normal(45000, 12000, n), np.where(plan == "postpago", rng.normal(25000, 8000, n), rng.normal(9000, 4000, n))).clip(2000), -2)
    reclamos = rng.poisson(0.7, n)
    datos = np.round(rng.gamma(2, 4, n), 1)
    edad = rng.integers(18, 80, n)
    cobertura_mala = np.isin(region, ["Aysén", "Magallanes", "Arica y Parinacota", "Tarapacá", "Los Ríos", "La Araucanía"])
    logit = -2.0 + 2.2 * cobertura_mala + 0.9 * reclamos - 0.02 * antig + 1.1 * (plan == "prepago") - 0.6 * (plan == "empresa") - 0.03 * datos + 0.8 * (antig < 6)
    churn = (rng.random(n) < 1 / (1 + np.exp(-logit))).astype(int)
    df = pd.DataFrame(dict(region=region, plan=plan, antiguedad_meses=antig, gasto_mensual=gasto, reclamos_12m=reclamos, datos_gb=datos, edad=edad, churn=churn))
    for c in ["datos_gb", "edad"]:                      # 8% gaps
        df.loc[rng.random(n) < 0.08, c] = np.nan
    return df

churn = simular_churn()
FEATS = [c for c in churn.columns if c != "churn"]
tr, te = churn.iloc[:2000], churn.iloc[2000:]
print(f"Customers: {len(churn)}   churn: {churn.churn.mean():.1%}   gaps: datos_gb {churn.datos_gb.isna().sum()}, edad {churn.edad.isna().sum()}")
print("\nChurn by region:")
print(churn.groupby("region").churn.mean().round(2).sort_values().to_string())
churn.head(8)



**Two preparations for the same data.** For `c50py`, the table as is: seven columns, two of
them text, with their gaps. For scikit-learn, the minimum it requires: the two text columns
converted into columns of zeros and ones (one per region and one per plan, 24 in total) and the gaps
filled with the training median. We compare with the same brake on both (minimum
customers per leaf), and we also search for scikit-learn's best hyperparameters by cross-validation,
so nobody can say the CART was poorly tuned.

We use the `c50py` defaults (pessimistic pruning with `cf = 0.25`) and only vary the
minimum customers per leaf, the same brake on both. We also add a row of `c50py` without
pruning, for a reason explained in Section 5: C4.5's pruning looks at the error, not at the ordering,
and for a risk ranking (AUC) it is better to turn it off.


In [ ]:

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV
import time

X_tr, X_te = tr[FEATS].values.astype(object), te[FEATS].values.astype(object)
Xd = pd.get_dummies(churn[FEATS], columns=["region", "plan"]).astype(float)
mediana = Xd.iloc[:2000].median(); Xd = Xd.fillna(mediana)
Xd_tr, Xd_te = Xd.iloc[:2000], Xd.iloc[2000:]
print("Columns for scikit-learn after dummies:", Xd.shape[1], "   for c50py:", len(FEATS))

def profundidad(nodo): return 0 if nodo.is_leaf else 1 + max(profundidad(nodo.children["left"]), profundidad(nodo.children["right"]))
filas = []
for msl in [10, 25]:
    t0 = time.time(); m = C5Classifier(categorical_features=["region", "plan"], min_samples_leaf=msl).fit(X_tr, tr.churn.values, feature_names=FEATS); t = time.time() - t0
    filas.append((f"c50py, default pruning, min_samples_leaf={msl}", "none", len(m.export_rules()), profundidad(m.tree_), m.score(X_te, te.churn.values), roc_auc_score(te.churn, m.predict_proba(X_te)[:, 1]), round(t, 1)))
    if msl == 25: c5_churn = m
    t0 = time.time(); s = DecisionTreeClassifier(min_samples_leaf=msl, random_state=0).fit(Xd_tr, tr.churn); t = time.time() - t0
    filas.append((f"scikit-learn, min_samples_leaf={msl}", "dummies + median", s.get_n_leaves(), s.get_depth(), s.score(Xd_te, te.churn), roc_auc_score(te.churn, s.predict_proba(Xd_te)[:, 1]), round(t, 1)))
    if msl == 25: sk_churn = s
t0 = time.time(); m = C5Classifier(categorical_features=["region", "plan"], min_samples_leaf=50, pruning=False).fit(X_tr, tr.churn.values, feature_names=FEATS); t = time.time() - t0
filas.append(("c50py, no pruning, min_samples_leaf=50 (to rank by risk)", "none", len(m.export_rules()), profundidad(m.tree_), m.score(X_te, te.churn.values), roc_auc_score(te.churn, m.predict_proba(X_te)[:, 1]), round(t, 1)))
gs = GridSearchCV(DecisionTreeClassifier(random_state=0), {"max_depth": [3, 4, 5, 6, 8, None], "min_samples_leaf": [10, 25, 50, 100]}, cv=5, scoring="roc_auc").fit(Xd_tr, tr.churn)
s = gs.best_estimator_
filas.append((f"scikit-learn, best by cross-validation {gs.best_params_}", "dummies + median", s.get_n_leaves(), s.get_depth(), s.score(Xd_te, te.churn), roc_auc_score(te.churn, s.predict_proba(Xd_te)[:, 1]), 0))
tabla = pd.DataFrame(filas, columns=["model", "preprocessing", "leaves", "depth", "test accuracy", "test AUC", "seconds"])
tabla["test accuracy"] = tabla["test accuracy"].round(3); tabla["test AUC"] = tabla["test AUC"].round(3)
print(tabla.to_string(index=False))
print(f"\nPredicting 'stays' for everyone: {1 - te.churn.mean():.3f}")



**How to read the table.** With the same brake, `c50py` gives a much smaller and better tree: with a minimum
of 25 per leaf, 12 leaves against 55, higher accuracy and higher AUC; with a minimum of 10, 19 leaves against 115. The
best scikit-learn by cross-validation reaches 33 leaves and ties in AUC with the 12-leaf C5.0,
with lower accuracy. AUC measures whether the model ranks customers by risk well; accuracy,
how many it classifies correctly with the 0.5 threshold. The unpruned row shows the other path: 27 leaves and
the best AUC among the single trees, because it keeps leaves that do not change the class but refine the
probability. And the preprocessing column is the one a business team notices first: nothing,
against dummies plus imputation.

**Why scikit-learn barely uses the region.** With 16 columns of zeros and ones, each one asks
"is it from region X?" and separates a small group from the rest: the impurity reduction of any of
them is small and the tree prefers splits on tenure or spend. C5.0 can ask about a
subset of regions in a single split, it groups the ones that behave alike, and the region becomes
the first question. The signal was in the data; what changed is whether the algorithm could
see it in one question. Compare the two trees with a minimum of 25 per leaf:


In [ ]:

print("c50py (12 leaves):\n")
c5_churn.print_tree(feature_names=FEATS)


In [ ]:

from sklearn.tree import export_text
print("scikit-learn (55 leaves), on the 24 columns:\n")
print(export_text(sk_churn, feature_names=list(Xd.columns)))
usa_region = sum(1 for f in sk_churn.tree_.feature if f >= 0 and Xd.columns[f].startswith("region_"))
print("Splits that use some region in the scikit-learn tree:", usa_region)



**The two trees drawn.** Both come from `plot_tree`, with the same arguments and the same box
format: `c50py.plot_tree` mirrors `sklearn.tree.plot_tree`. The only difference inside the boxes is the
impurity measure: entropy in C5.0, Gini in CART (hidden here with `impurity=False`). The colours differ on
purpose: `c50py` uses its own palette (teal and violet) so you can tell which package drew each tree.
Look at the size and at what each one
asks: region and complaints at the top in the C5.0 tree; tenure, spend and age in the scikit-learn one,
with the regions one at a time and at the bottom.


In [ ]:

import matplotlib.pyplot as plt
from sklearn.tree import plot_tree

# The c50py tree, with the same function and the same arguments as the scikit-learn one
fig, ax = plt.subplots(figsize=(20, 8))
c5_churn.plot_tree(feature_names=FEATS, class_names=["stays", "leaves"], filled=True, fontsize=8, ax=ax, impurity=False)
ax.set_title("C5.0 (c50py), minimum 25 per leaf")
plt.show()

fig, ax = plt.subplots(figsize=(28, 10))
plot_tree(sk_churn, feature_names=list(Xd.columns), class_names=["stays", "leaves"], filled=True, fontsize=6, ax=ax, impurity=False)
ax.set_title("CART (scikit-learn), minimum 25 per leaf")
plt.show()
ax.set_title("scikit-learn, minimum 25 per leaf, on the 24 columns: 55 leaves")
plt.show()



**The rule that fired, customer by customer.** This is what turns the tree into a
retention tool: `predict_rule` returns, for each customer, the chain of conditions that
took them to their leaf. Each leaf is a segment, and each segment suggests a different action: a new
customer from a region with good coverage is not offered the same thing as one from Tarapacá with
two complaints. Notice that a missing value can appear as a condition (`datos_gb MISSING`):
C5.0 did not impute it, it used it as what it is.


In [ ]:

rng = np.random.default_rng(7)
idx = rng.choice(len(te), 10, replace=False)
reglas = c5_churn.predict_rule(X_te[idx], feature_names=FEATS)
p_fuga = c5_churn.predict_proba(X_te[idx])[:, 1]
for k, i in enumerate(idx):
    c = te.iloc[i]
    print(f"Customer {2000 + i + 1}: {c.region}, {c.plan}, {int(c.antiguedad_meses)} months, {int(c.reclamos_12m)} complaints, data {c.datos_gb}, left: {'yes' if c.churn else 'no'}")
    print(f"   P(leaves) = {p_fuga[k]:.3f}   rule: {reglas[k]}\n")


In [ ]:

# How many test customers fall under each rule, and what fraction left: the retention segments
segmentos = pd.DataFrame({"regla": c5_churn.predict_rule(X_te, feature_names=FEATS), "se fue": te.churn.values})
resumen = segmentos.groupby("regla")["se fue"].agg(["count", "mean"]).rename(columns={"count": "test customers", "mean": "fraction that left"}).sort_values("fraction that left", ascending=False).round(2)
pd.set_option("display.max_colwidth", 200)
print(resumen.to_string())



**And if all that matters is getting it right.** `trials = 10` trains ten trees in sequence with the
AdaBoost scheme from Notebook 4, without changing anything else, and we compare it with scikit-learn's gradient boosting
on the preprocessed columns.


In [ ]:

t0 = time.time(); c5_boost = C5Classifier(categorical_features=["region", "plan"], min_samples_leaf=25, trials=10, random_state=0).fit(X_tr, tr.churn.values, feature_names=FEATS); t_c5 = time.time() - t0
t0 = time.time(); gb = GradientBoostingClassifier(n_estimators=200, max_depth=3, learning_rate=0.05, random_state=0).fit(Xd_tr, tr.churn); t_gb = time.time() - t0
print(pd.DataFrame([
    ("c50py, trials = 10, min_samples_leaf = 25", "none", c5_boost.score(X_te, te.churn.values), roc_auc_score(te.churn, c5_boost.predict_proba(X_te)[:, 1]), round(t_c5, 1)),
    ("scikit-learn GradientBoostingClassifier (200, depth 3, rate 0.05)", "dummies + median", gb.score(Xd_te, te.churn), roc_auc_score(te.churn, gb.predict_proba(Xd_te)[:, 1]), round(t_gb, 1)),
], columns=["model", "preprocessing", "test accuracy", "test AUC", "seconds"]).round(3).to_string(index=False))



Even on AUC, in exchange for losing the per-customer rule. It is the usual decision between explaining
and getting it right, and C5.0 leaves it in one parameter. The cost that does show is time: `c50py` is
written in pure Python and is slower than scikit-learn's C; on tables of tens of thousands of
rows, `numeric_threshold_strategy="quantile"` (32 quantiles instead of all thresholds) and
patience are advisable.

The sections that follow open up the mechanism of each of the four things, with the usual small tables
so you can count on your fingers.



## 2. Categories without dummies, and the gain ratio

Let us go back to the 14 firms of Notebook 2 to see the mechanism with numbers that can be counted. ID3 opened one branch per value of the attribute: low,
medium and high debt were three branches. C5.0 does something different: it tries every way of separating
the values into **two** groups ({low} against {medium, high}, {medium} against {low, high}, and so on) and
picks the best. With $k$ values there are $2^{k-1} - 1$ binary partitions; `c50py` tries all of them
up to `max_categories_exhaustive` (12 by default) and beyond that uses a heuristic.

To compare splits that break the data into different numbers of pieces, C4.5 divides the gain by the
entropy of the partition (*split information*):

$$
\text{gain ratio}(S) = \frac{H(Y) - \sum_g \frac{n_g}{n} H(Y \text{ en } g)}{-\sum_g \frac{n_g}{n} \log_2 \frac{n_g}{n}}
$$

The numerator is the information gain of Notebook 1; the denominator is the entropy of the
sizes of the groups $g$ formed by split $S$ ($n_g$ observations in the group, $n$ in total):
it is 1 bit if the split breaks the data into two halves, 1.58 if into three thirds, and it grows with the number
of pieces. Dividing penalizes splits that win "for free" by splitting a lot (the identifier
trap).


In [ ]:

# The 14 firms that applied for credit (the small table of the course)
credito = pd.DataFrame({
    "tamano":    ["pequena", "grande", "mediana", "pequena", "grande", "mediana", "pequena", "grande", "mediana", "pequena", "grande", "mediana", "pequena", "grande"],
    "deuda":     ["baja", "baja", "baja", "baja", "alta", "alta", "alta", "alta", "alta", "media", "media", "media", "media", "media"],
    "garantia":  ["no", "si", "no", "si", "si", "si", "no", "no", "no", "no", "si", "si", "no", "no"],
    "historial": ["bueno", "malo", "malo", "bueno", "bueno", "malo", "bueno", "malo", "bueno", "bueno", "bueno", "malo", "malo", "malo"],
    "paga":      ["si", "si", "si", "si", "si", "si", "no", "no", "no", "si", "si", "no", "no", "no"],
})
credito.index = range(1, 15)
credito


In [ ]:

def H(serie):
    p = pd.Series(serie).value_counts(normalize=True)
    return float(-(p * np.log2(p)).sum())

def ganancia_y_ratio(df, mascara_por_grupo):
    # mascara_por_grupo: list of boolean masks, one per group of the split
    Hy = H(df["paga"]); n = len(df)
    gan = Hy - sum(m.sum() / n * H(df.loc[m, "paga"]) for m in mascara_por_grupo)
    split_info = -sum(m.sum() / n * np.log2(m.sum() / n) for m in mascara_por_grupo if m.sum() > 0)
    return gan, split_info, gan / split_info if split_info > 0 else 0.0

filas = []
for a in ["tamano", "deuda", "garantia", "historial"]:
    vals = sorted(credito[a].unique())
    g, s, r = ganancia_y_ratio(credito, [credito[a] == v for v in vals])
    filas.append((a, "all values (ID3): " + " / ".join(vals), round(g, 3), round(s, 3), round(r, 3)))
    vistos = set()
    for k in range(1, len(vals) // 2 + 1):
        for S in itertools.combinations(vals, k):
            resto = tuple(v for v in vals if v not in S)
            if frozenset(resto) in vistos: continue
            vistos.add(frozenset(S))
            m = credito[a].isin(S)
            g, s, r = ganancia_y_ratio(credito, [m, ~m])
            filas.append((a, "{" + ", ".join(S) + "} against {" + ", ".join(resto) + "}", round(g, 3), round(s, 3), round(r, 3)))
print(pd.DataFrame(filas, columns=["attribute", "split", "gain", "entropy of the partition", "gain ratio"]).to_string(index=False))



Look at the debt row: splitting {baja} against {media, alta} has exactly the same gain as
splitting into three (0.292 bits), because all the information in debt was in separating the four
low-debt firms, all of which pay. But the binary partition splits 4 against 10, with
entropy 0.863, and the three-value one has entropy 1.577: the gain ratio of the binary split almost
doubles ID3's (0.338 against 0.185). C5.0 gets the same thing with fewer pieces, and it has 10
firms left together to keep asking questions.

Now the full tree with `c50py`. We tell it which columns are categorical (or ask it to
infer them with `infer_categorical=True`) and pass it the names so the tree is readable.


In [ ]:

ATRIBUTOS = ["tamano", "deuda", "garantia", "historial"]
Xc = credito[ATRIBUTOS].values.astype(object); yc = credito["paga"].values

c5_14 = C5Classifier(categorical_features=ATRIBUTOS, pruning=False, min_samples_leaf=1)
c5_14.fit(Xc, yc, feature_names=ATRIBUTOS)
c5_14.print_tree(feature_names=ATRIBUTOS)
print()
print("Rules, one per leaf:")
for regla in c5_14.export_rules(feature_names=ATRIBUTOS):
    print("  ", regla)



Eight leaves for 14 firms, against ID3's five: the root is the same question written in
binary, and the rest is distributed differently because after "deuda in {baja}" there are 10 firms
left mixed together instead of two groups of 5. Like ID3, C5.0 without a brake splits until every group is
pure, and with 14 firms that is memorizing.

**The honest comparison with CART.** scikit-learn needs dummies (one column of zeros and ones per
category). With validation leaving one firm out each time (*leave-one-out*), which is the only
thing you can do with 14 rows:


In [ ]:

from sklearn.model_selection import LeaveOneOut
Xd = pd.get_dummies(credito[ATRIBUTOS]).values.astype(float)
ac_c5, ac_cart = [], []
for tr, te in LeaveOneOut().split(Xd):
    m = C5Classifier(categorical_features=ATRIBUTOS, min_samples_leaf=1).fit(Xc[tr], yc[tr], feature_names=ATRIBUTOS); ac_c5.append(m.predict(Xc[te])[0] == yc[te][0])
    s = DecisionTreeClassifier(random_state=0).fit(Xd[tr], yc[tr]); ac_cart.append(s.predict(Xd[te])[0] == yc[te][0])
print(f"Leave-one-out accuracy: c50py {np.mean(ac_c5):.3f}   CART with dummies {np.mean(ac_cart):.3f}")
print("Leaves of the CART with dummies on the 14:", DecisionTreeClassifier(random_state=0).fit(Xd, yc).get_n_leaves())



C5.0 with its default pruning gets 10 out of 14 right (71%); the CART with dummies, 8 out of 14 (57%), the same
as predicting "pays" for all of them. With 14 firms this is an anecdote, not evidence: two
firms of difference. What can be stated is the reading: the C5.0 tree speaks of
"deuda in {baja}" and not of "deuda_baja == 1", and C4.5's pruning leaves it smaller than the eight
unrestrained leaves above.



## 3. The 219 firms: the `c50py` tree against the CART of Notebook 2

With numeric ratios, C5.0 searches for thresholds just like CART, with gain ratio instead of Gini (and with
two more C4.5 rules: only attributes with at least average gain compete on gain ratio,
and continuous variables pay a small penalty for the number of thresholds they offer).
We use the default values.


In [ ]:

# The 219 firms with financial ratios (the default table of the course, with 5 of its ratios).
# It is pasted right here so the notebook does not depend on any external file.
from io import StringIO
IMPAGO_CSV = """deuda_activos,razon_corriente,ventas_deuda,ln_activos,roa,impago
0.374,2.337,1.436,14.697,0.073,0.0
0.435,2.638,1.385,14.877,0.042,0.0
0.443,2.099,0.452,14.931,0.016,0.0
0.23,2.506,4.217,15.586,-0.041,0.0
0.317,2.273,3.116,15.708,0.021,0.0
0.312,3.282,4.842,15.785,0.047,0.0
0.628,1.386,2.91,14.066,0.064,0.0
0.64,1.777,2.895,14.236,0.068,0.0
0.719,1.44,1.505,14.697,0.075,0.0
0.551,1.078,1.944,15.564,0.113,0.0
0.541,0.991,2.091,15.581,0.022,0.0
0.575,0.995,1.787,15.738,0.035,0.0
0.454,3.266,7.321,14.877,0.064,0.0
0.574,3.017,3.159,14.457,0.07,0.0
0.526,3.333,1.962,14.399,0.077,0.0
0.724,0.8,1.625,14.036,0.016,0.0
0.508,1.406,4.356,14.349,0.406,0.0
0.494,1.051,2.299,14.515,0.117,0.0
0.572,1.557,2.106,14.016,0.126,0.0
0.578,1.498,2.033,14.293,0.086,0.0
0.558,1.605,2.035,14.52,0.099,0.0
0.314,2.127,10.633,13.738,0.087,0.0
0.467,1.661,5.599,14.219,0.094,0.0
0.499,1.482,3.109,14.424,0.067,0.0
0.259,3.734,5.692,14.172,0.234,0.0
0.202,4.804,7.312,14.417,0.195,0.0
0.268,3.663,6.08,14.897,0.228,0.0
0.409,1.671,4.523,14.253,0.12,0.0
0.307,2.171,6.817,14.38,0.172,0.0
0.258,3.481,10.557,14.454,0.274,0.0
0.393,1.55,3.81,14.68,0.024,0.0
0.558,1.102,2.597,14.108,0.005,0.0
0.432,1.365,3.534,14.01,0.119,0.0
0.53,1.281,3.099,12.687,0.272,0.0
0.244,3.411,16.602,13.143,0.489,0.0
0.354,2.877,3.112,13.311,0.075,0.0
0.381,1.56,2.749,14.696,0.074,0.0
0.411,2.198,2.46,14.839,0.043,0.0
0.401,1.954,2.067,14.915,0.037,0.0
0.387,1.955,2.616,14.835,0.342,0.0
0.438,1.029,2.619,14.929,0.023,0.0
0.413,1.44,2.869,14.889,-0.011,0.0
0.891,1.118,2.023,13.415,0.036,0.0
1.052,0.741,1.8,14.355,-0.093,0.0
0.466,0.899,1.509,14.364,0.034,0.0
0.491,1.196,2.335,14.414,-0.017,0.0
0.505,1.053,2.678,14.631,0.076,0.0
0.354,2.45,3.08,14.453,0.084,0.0
0.464,1.843,2.521,14.559,0.119,0.0
0.566,1.286,1.675,14.838,0.111,0.0
0.543,1.766,2.619,14.933,0.029,0.0
0.855,1.131,1.805,15.005,0.029,0.0
0.558,1.744,2.516,15.074,0.028,0.0
0.736,1.153,1.564,14.292,0.054,0.0
0.708,1.192,1.352,14.432,0.034,0.0
0.657,1.269,1.395,14.387,0.027,0.0
0.653,1.061,1.588,14.303,0.04,0.0
0.619,1.005,2.113,14.235,0.038,0.0
0.197,9.632,5.15,14.586,0.032,0.0
0.284,1.461,6.585,13.479,0.11,0.0
0.3,1.438,5.251,13.541,0.012,0.0
0.283,2.928,5.778,13.561,0.016,0.0
0.476,1.788,5.329,13.143,0.123,0.0
0.47,1.759,5.033,13.329,0.13,0.0
0.459,2.104,4.49,13.372,0.076,0.0
0.388,2.409,5.319,13.283,0.201,0.0
0.547,1.979,2.09,13.806,0.15,0.0
0.568,1.812,1.937,13.914,0.071,0.0
0.461,1.923,1.499,14.059,-0.013,0.0
0.538,2.174,1.801,14.208,0.025,0.0
0.497,1.878,1.917,14.284,0.075,0.0
0.208,3.306,5.181,15.336,-0.003,0.0
0.241,3.005,4.709,15.428,0.011,0.0
0.222,3.11,3.776,15.427,0.021,0.0
0.392,2.268,6.865,12.39,0.334,0.0
0.252,5.229,7.751,12.318,0.176,0.0
0.849,1.56,1.326,12.304,-0.186,0.0
0.36,2.671,5.196,12.142,0.186,0.0
0.028,34.514,71.252,12.295,0.292,0.0
0.631,1.369,1.835,13.284,0.136,0.0
0.507,1.464,1.625,13.732,-0.217,0.0
0.578,1.577,2.553,13.963,0.02,0.0
0.448,1.681,0.566,13.768,0.024,0.0
0.363,2.721,2.762,13.301,0.446,0.0
0.403,2.199,2.673,13.506,0.067,0.0
0.4,2.237,1.139,13.576,0.043,0.0
0.432,1.579,2.39,14.7,0.066,0.0
0.464,1.763,2.067,14.843,0.059,0.0
0.475,1.707,1.833,14.95,0.06,0.0
0.089,8.849,16.344,15.39,0.013,0.0
0.125,7.175,11.171,15.513,0.059,0.0
0.125,7.156,12.795,15.55,0.041,0.0
0.539,1.764,3.733,15.384,0.088,0.0
0.502,1.79,3.797,15.397,0.035,0.0
0.522,1.572,2.624,15.391,0.029,0.0
0.77,0.383,1.984,15.834,-0.046,0.0
0.746,0.465,3.31,16.041,0.057,0.0
0.666,0.539,3.919,16.051,0.018,0.0
0.14,8.346,5.975,15.361,0.008,0.0
0.183,5.927,4.699,15.443,0.026,0.0
0.24,4.46,3.828,15.549,0.016,0.0
0.749,1.39,2.739,15.184,0.015,0.0
0.72,1.212,2.976,15.14,0.006,0.0
0.824,0.937,3.12,15.016,-0.089,0.0
0.127,0.463,3.018,13.706,0.036,0.0
0.584,3.254,1.629,14.581,0.053,0.0
0.565,0.66,1.621,14.753,0.085,0.0
0.853,3.41,0.45,13.939,0.012,0.0
0.836,4.713,0.337,14.072,0.007,0.0
0.968,2.708,0.249,14.346,-0.016,0.0
0.387,2.077,1.735,15.354,-0.039,0.0
0.398,2.395,2.933,15.532,0.059,0.0
0.327,2.428,1.813,15.518,0.036,0.0
0.635,1.999,0.061,15.292,-0.005,0.0
0.279,4.784,0.637,14.664,0.016,0.0
0.162,2.572,1.5,14.58,0.057,0.0
0.21,1.923,15.689,12.954,0.112,0.0
0.434,0.844,4.415,12.837,-0.246,0.0
0.271,1.323,5.378,12.849,0.17,0.0
0.416,2.122,5.392,13.508,0.098,0.0
0.409,2.225,4.438,13.477,0.018,0.0
0.262,3.185,5.914,13.392,0.128,0.0
0.23,4.172,5.796,14.956,0.218,0.0
0.085,11.333,15.115,15.14,0.246,0.0
0.134,6.66,9.398,15.292,0.101,0.0
0.759,2.52,2.074,13.84,0.084,0.0
0.77,1.92,1.572,14.162,0.031,0.0
0.675,2.107,1.621,13.957,0.036,0.0
0.601,2.387,1.541,13.442,-0.214,0.0
0.733,1.376,1.151,13.28,-0.11,0.0
0.822,1.461,1.286,13.478,-0.026,0.0
0.15,9.243,7.907,13.92,0.068,0.0
0.103,13.87,11.417,14.067,0.135,0.0
0.054,35.477,17.746,14.134,0.107,0.0
0.672,1.436,1.373,13.419,0.145,1.0
0.292,0.205,1.514,13.547,0.171,1.0
0.614,1.341,1.041,13.703,0.069,1.0
0.507,1.49,3.369,13.853,0.203,1.0
0.7,1.17,0.148,14.649,0.092,1.0
0.616,2.062,1.508,14.571,0.074,1.0
0.622,1.136,0.098,14.945,0.043,1.0
0.731,1.134,0.852,15.429,0.041,1.0
0.747,0.97,0.859,15.636,0.037,1.0
0.485,1.984,3.443,13.503,0.059,1.0
0.599,1.421,2.111,13.788,0.033,1.0
0.685,1.229,1.801,13.965,0.009,1.0
0.675,0.869,1.083,13.874,0.096,1.0
0.828,0.878,0.992,14.654,0.018,1.0
0.696,1.079,3.177,14.413,0.072,1.0
1.314,0.703,0.86,12.145,0.072,1.0
1.458,0.588,0.01,11.974,-0.086,1.0
1.732,0.454,0.0,11.772,-0.171,1.0
0.743,0.921,1.933,13.546,0.185,1.0
0.752,1.488,2.063,13.671,0.202,1.0
1.074,1.367,1.418,13.605,0.13,1.0
0.297,0.154,0.94,16.088,0.064,1.0
0.268,1.354,0.861,16.111,0.025,1.0
0.09,1.484,2.472,16.149,0.019,1.0
0.405,3.01,2.249,15.232,0.022,1.0
0.488,2.13,2.073,15.416,0.03,1.0
0.528,1.87,2.078,15.514,0.071,1.0
0.715,1.106,1.3,15.805,0.055,1.0
0.66,1.287,1.822,15.719,0.018,1.0
0.678,1.252,1.754,15.89,0.029,1.0
0.631,1.578,1.041,14.786,0.02,1.0
0.635,1.733,1.055,14.82,0.026,1.0
0.659,1.585,1.171,14.771,0.028,1.0
0.689,1.368,0.95,12.36,-0.033,1.0
0.464,1.238,1.069,12.662,-0.162,1.0
0.845,1.356,0.552,12.717,-0.352,1.0
0.548,1.741,0.697,16.077,0.002,1.0
0.595,1.234,0.758,16.058,0.0,1.0
0.768,0.919,0.355,16.536,-0.08,1.0
0.535,3.214,3.409,12.725,0.186,1.0
0.588,2.784,4.476,12.623,0.303,1.0
0.683,2.317,4.213,12.453,0.327,1.0
0.42,1.553,1.755,15.717,0.012,1.0
0.397,2.599,2.095,15.712,0.028,1.0
0.348,2.223,1.268,15.703,0.021,1.0
0.592,1.518,2.17,14.555,0.047,1.0
0.608,1.401,1.509,14.678,0.014,1.0
0.661,0.001,0.941,14.767,-0.013,1.0
1.48,0.225,0.001,12.488,-0.078,1.0
1.575,0.324,0.044,12.488,-0.009,1.0
0.638,0.87,1.662,15.163,-0.005,1.0
0.624,0.906,1.764,15.229,0.03,1.0
0.631,1.037,1.033,8.626,0.086,1.0
0.27,2.783,10.977,13.105,0.19,1.0
0.471,2.03,6.381,13.449,0.224,1.0
0.292,3.383,9.848,13.567,0.22,1.0
0.701,1.253,3.098,15.845,0.056,1.0
0.799,0.047,2.534,15.942,0.048,1.0
0.772,1.24,2.471,16.078,0.054,1.0
1.181,0.804,0.944,12.801,0.075,1.0
0.464,1.293,2.927,12.743,0.092,1.0
0.514,1.105,2.764,12.839,0.082,1.0
0.386,2.301,6.078,12.668,0.399,1.0
0.723,1.327,1.854,13.612,0.125,1.0
0.695,1.38,1.813,13.589,0.085,1.0
0.638,1.322,4.07,13.505,0.286,1.0
0.555,1.614,3.204,13.949,0.13,1.0
0.594,1.528,2.718,14.176,0.125,1.0
1.401,0.709,0.208,12.076,-0.146,1.0
0.916,0.616,1.27,12.629,0.316,1.0
0.75,0.489,2.46,12.725,0.29,1.0
0.445,9.164,0.0,11.167,-0.737,1.0
0.767,0.664,1.149,12.778,-0.779,1.0
1.314,0.703,0.86,12.145,0.072,1.0
1.458,0.588,0.01,11.974,-0.086,1.0
1.732,0.454,0.0,11.772,-0.171,1.0
0.775,1.025,3.294,13.079,0.112,1.0
0.731,0.932,4.179,12.889,0.126,1.0
0.842,0.673,1.911,13.274,-0.04,1.0
0.733,1.226,2.588,13.42,0.091,1.0
0.707,1.325,2.385,13.461,0.06,1.0
1.209,0.656,1.554,13.272,-0.565,1.0
0.507,1.49,3.369,13.853,0.203,1.0
0.7,1.17,1.479,14.649,0.092,1.0
0.616,2.062,2.277,14.571,0.074,1.0"""
impago = pd.read_csv(StringIO(IMPAGO_CSV))
print("Firms:", len(impago), "  Fraction in default:", round(impago["impago"].mean(), 3))
impago.describe().round(3).T[["mean", "min", "50%", "max"]]


In [ ]:

# The same partition in every notebook of the set: 146 firms to train, 73 to test
RATIOS = ["deuda_activos", "razon_corriente", "ventas_deuda", "ln_activos", "roa"]
rng = np.random.default_rng(2)
orden = rng.permutation(len(impago))
es_prueba = np.zeros(len(impago), dtype=bool); es_prueba[orden[:73]] = True
impago["conjunto"] = np.where(es_prueba, "prueba", "entrenamiento")

X = impago[RATIOS]; y = impago["impago"]
X_train, y_train = X[~es_prueba], y[~es_prueba]
X_test, y_test = X[es_prueba], y[es_prueba]
print("Training:", len(X_train), "  Test:", len(X_test))
print("Fraction in default: training", round(y_train.mean(), 3), " test", round(y_test.mean(), 3))


In [ ]:

Xtr, Xte = X_train.values.astype(float), X_test.values.astype(float)

def c5(**kw):
    m = C5Classifier(**kw)
    m.fit(Xtr, y_train.values, feature_names=RATIOS)
    return m

filas = []
for msl in [1, 5, 10, 20]:
    m = c5(min_samples_leaf=msl)
    filas.append((msl, len(m.export_rules(feature_names=RATIOS)), round(m.score(Xtr, y_train), 3), round(m.score(Xte, y_test), 3)))
print(pd.DataFrame(filas, columns=["min_samples_leaf", "leaves", "training accuracy", "test accuracy"]).to_string(index=False))
print()
cart2 = DecisionTreeClassifier(max_depth=2, random_state=0).fit(X_train, y_train)
print(f"Reference, depth-2 CART (Notebook 2): training {cart2.score(X_train, y_train):.3f}   test {cart2.score(X_test, y_test):.3f}")


In [ ]:

c5_def = c5()          # default values: minimum 1 per leaf, pessimistic pruning with cf = 0.25
c5_def.print_tree()



Six leaves with the default values, 82.9% on training and 76.7% on test. The root and the
first level are the same as in the CART of Notebook 2 (sales over debt 1.278; then current
ratio 1.866 and debt over assets 0.586): gain ratio and Gini agree on the first two
questions and differ in what they do with the remaining groups. On these 73 test firms the
two-level CART gets 82.2% right and this tree 76.7%: four firms of difference, which with so
little data do not say which criterion is better.

What is different is the output. `export_rules()` writes each leaf as a rule, and
`predict_rule()` returns, for a specific firm, the rule that was applied to it: exactly
what a credit committee asks for when it asks "why".


In [ ]:

for regla in c5_def.export_rules():
    print(regla)
print()
i = 0
print("Test firm", X_test.index[i], "  ratios:", dict(X_test.iloc[i].round(3)))
print("Rule applied:", c5_def.predict_rule(Xte[[i]])[0])
print("Prediction:", c5_def.predict(Xte[[i]])[0], "  actual:", y_test.iloc[i])



## 4. Missing values: the firm goes down both branches

In practice a firm may be missing a ratio. CART needs someone to fill the gap
beforehand, typically with the median. C5.0 does not: when training, if a firm is missing the variable a
node asks about, it sends it down **both branches**, with a weight equal to the fraction of the
known firms that went each way. That is why in trees trained on incomplete data
the leaves show counts with decimals: whole firms plus pieces of firms. When predicting,
the same happens:

$$
P(\text{impago} \mid x) = \frac{n_{\text{izq}}}{n_{\text{izq}} + n_{\text{der}}} P_{\text{izq}} + \frac{n_{\text{der}}}{n_{\text{izq}} + n_{\text{der}}} P_{\text{der}}
$$

where $n_{\text{izq}}$ and $n_{\text{der}}$ are how many training firms with the variable
known went each way (left and right), and $P_{\text{izq}}$, $P_{\text{der}}$ what each branch says when you keep
going down with the ratios that are known. When choosing the split there is also a correction: the
gain is computed with the firms that have the value and multiplied by the known fraction.

We erase 20% of the values at random, in training and in test, and compare.


In [ ]:

rng = np.random.default_rng(4)
Xm = Xtr.copy(); Xm[rng.random(Xm.shape) < 0.20] = np.nan
Xtm = Xte.copy(); Xtm[rng.random(Xtm.shape) < 0.20] = np.nan
print("Missing values in training:", int(np.isnan(Xm).sum()), "of", Xm.size, "  complete rows:", int((~np.isnan(Xm).any(axis=1)).sum()), "of 146")

c5_falt = C5Classifier(min_samples_leaf=10).fit(Xm, y_train.values, feature_names=RATIOS)
c5_falt.print_tree(feature_names=RATIOS)


In [ ]:

mediana = np.nanmedian(Xm, axis=0)
Xi, Xti = np.where(np.isnan(Xm), mediana, Xm), np.where(np.isnan(Xtm), mediana, Xtm)
completas = ~np.isnan(Xm).any(axis=1)

comparacion = pd.DataFrame([
    ("c50py, fractional propagation", C5Classifier(min_samples_leaf=10).fit(Xm, y_train.values).score(Xtm, y_test.values)),
    ("CART, gaps filled with the median", DecisionTreeClassifier(min_samples_leaf=10, random_state=0).fit(Xi, y_train).score(Xti, y_test)),
    ("CART, complete rows only", DecisionTreeClassifier(min_samples_leaf=10, random_state=0).fit(Xm[completas], y_train.values[completas]).score(Xti, y_test)),
    ("c50py without gaps (reference)", C5Classifier(min_samples_leaf=10).fit(Xtr, y_train.values).score(Xte, y_test.values)),
    ("CART without gaps (reference)", DecisionTreeClassifier(min_samples_leaf=10, random_state=0).fit(Xtr, y_train).score(Xte, y_test)),
], columns=["model", "test accuracy"]).round(3)
print(comparacion.to_string(index=False))



The counts with decimals in the leaves are fractional propagation in plain sight: with 20% of
gaps, pruning leaves a two-leaf tree, because the information that remains is scarce and spread out.
As for accuracy, with 73 test firms the differences are two to six firms and do not
allow declaring a winner: filling with the median is not a bad option on a small table,
and discarding incomplete rows is when few remain (46 out of 146). The advantage of
fractional propagation is that it does not invent a value: it preserves the uncertainty and shows it in the
prediction.



## 5. Pessimistic pruning: pruning without validation data

The cost-complexity pruning of Notebook 3 needs data the tree did not see. Quinlan prunes
using training data only: he recognizes that a leaf's training error is optimistic and
corrects it upwards. A leaf with $N$ cases and $E$ errors is treated as a binomial sample, and it is
charged the highest value of the true error that is still compatible with what was observed at confidence
level `cf`. In the C4.5 code that quantity is called `AddErrs`:

$$
E_{\text{pes}} = E + \text{AddErrs}(N, E), \qquad \text{AddErrs}(N, 0) = N\,(1 - cf^{1/N})
$$

and for $E > 0$ it is the upper limit of the binomial interval with the $z$ from the C4.5 table (0.69
for the default 0.25; 1.28 for 0.10; 2.33 for 0.01). The important part is the formula for
$E = 0$: a **pure** leaf pays too. A leaf with a single observation and no errors receives 0.75
pessimistic errors with `cf = 0.25`; one with three, 1.11. That is the mechanism that makes it possible to prune leaves
that only memorized. For each internal node the pessimistic errors of its leaves are summed and
compared with those of the node turned into a leaf; if as a leaf it is no worse (with a tolerance of 0.1),
it is pruned, from the bottom up.

This is the pruning of `c50py` version 0.3.0, which follows the C4.5 code. The previous version
used a normal approximation that left pure leaves unpunished: the same tree on the 146
firms ended up with 32 leaves and 100% on training. Now:


In [ ]:

filas = []
for cf in [0.01, 0.10, 0.25, 0.50]:
    for msl in [1, 5]:
        m = c5(min_samples_leaf=msl, cf=cf)
        filas.append((cf, msl, len(m.export_rules()), round(m.score(Xtr, y_train), 3), round(m.score(Xte, y_test), 3)))
m = c5(min_samples_leaf=1, pruning=False)
filas.append(("no pruning", 1, len(m.export_rules()), round(m.score(Xtr, y_train), 3), round(m.score(Xte, y_test), 3)))
print(pd.DataFrame(filas, columns=["cf", "min_samples_leaf", "leaves", "training accuracy", "test accuracy"]).to_string(index=False))



With these data the unpruned tree is already small (the C4.5 rules for choosing splits stop it
earlier), so pruning has little to do; the most demanding `cf` (0.01) removes one leaf. The calculation
by hand, on a real node of the tree: "debt over assets <= 0.374", with 4 firms (3 pay, 1 in
default) split into a pure leaf of 1 and a pure leaf of 3.


In [ ]:

from c50py.tree import _add_errs          # the AddErrs function of C4.5, as it is in the package
for cf in [0.25, 0.10, 0.05, 0.01]:
    como_hoja = 1 + _add_errs(4, 1, cf)                          # the node as a leaf: 1 observed error
    subarbol = _add_errs(1, 0, cf) + _add_errs(3, 0, cf)         # two pure leaves: 0 observed errors
    print(f"cf = {cf:.2f}   node as leaf {como_hoja:.2f}   subtree {subarbol:.2f}   ->", "pruned" if como_hoja <= subarbol + 0.1 else "kept")



With `cf = 0.25` the subtree is kept (2.19 against 1.86: the two pure leaves pay 0.75 and 1.11);
with `cf = 0.01` the penalty on small leaves grows faster than the node's and it is pruned. It is
exactly what the previous table shows.

**A property of this pruning worth keeping in mind.** It looks at the error, not at the ordering. In an
imbalanced problem like churn (20% churn), many leaves predict the majority class
and only refine the probability; for the classification error they contribute nothing, and C4.5 removes them.
That is why in Section 1 the unpruned tree ranked customers better (AUC 0.767) than the pruned one
(0.716), even though its accuracy was lower. If the use is a risk ranking, `pruning=False` with a reasonable minimum
per leaf; if the use is rules and decisions, the default pruning.



## 6. Built-in boosting

C5.0 has boosting inside: `trials = 10` trains ten trees in sequence with the
AdaBoost scheme from Notebook 4. Each tree is trained with per-firm weights, its weighted
error $\varepsilon$ is computed, its weight in the vote is $\alpha = \tfrac{1}{2} \ln \frac{1 - \varepsilon}{\varepsilon}$,
the misclassified firms are multiplied by $e^{\alpha}$, the weights are renormalized and it continues. It
stops early if a tree is perfect (there is nothing to correct) or if it is worse than a coin flip.

With the default values the first tree gets 82.9% right on training, so it has
errors left to correct and boosting has work to do from the start.


In [ ]:

filas = []
for t in [1, 3, 5, 10, 25]:
    b = c5(trials=t, random_state=0)
    n_arboles = len(b.ensemble_) if hasattr(b, "ensemble_") else 1
    filas.append((t, n_arboles, round(b.score(Xtr, y_train), 3), round(b.score(Xte, y_test), 3)))
print(pd.DataFrame(filas, columns=["trials", "trees trained", "training accuracy", "test accuracy"]).to_string(index=False))

b10 = c5(trials=10, random_state=0)
print("\nAlphas of the ten rounds:", np.round(b10.alphas_, 3))
print("Weighted errors they imply (e = 1 / (1 + exp(2 alpha))):", np.round(1 / (1 + np.exp(2 * np.array(b10.alphas_))), 3))



Ten rounds take the test accuracy from 76.7% to 82.2%, the same as the depth-2 CART,
and training to 100%: from there on there is nothing left to correct and more rounds change
nothing. As in Notebook 4, the number of rounds is a knob that should be chosen with validation.

## 7. There is also a `C5Regressor`

The same interface for numeric responses, with variance as impurity. We predict `roa` from
the other four ratios, as in Notebook 2.


In [ ]:

SIN_ROA = [r for r in RATIOS if r != "roa"]
reg = C5Regressor(min_samples_leaf=10, numeric_threshold_strategy="all")
reg.fit(X_train[SIN_ROA].values.astype(float), X_train["roa"].values, feature_names=SIN_ROA)
reg.print_tree(feature_names=SIN_ROA) if hasattr(reg, "print_tree") else None
ref = DecisionTreeRegressor(min_samples_leaf=10, random_state=0).fit(X_train[SIN_ROA], X_train["roa"])
print(f"\nTest R2: c50py {reg.score(X_test[SIN_ROA].values.astype(float), X_test['roa'].values):.3f}   CART {ref.score(X_test[SIN_ROA], X_test['roa']):.3f}")



## 8. To close

C5.0 and CART are two answers to the same questions: how to choose the split, what to do with
categories and with gaps, when to stop. Neither is "the correct one". On 219 firms the two give
different trees and accuracies that cannot be told apart with 73 test firms; what does set C5.0 apart is
what it delivers around the tree: categories without dummies, gaps without imputation, rules as text and
the exact rule for each prediction. Knowing both branches is what lets you read with judgment what
any library delivers.

## Exercises

1. **Gain ratio and the identifier.** Add to the 14 firms an `id` column with a different value
   per firm and compute, with `ganancia_y_ratio`, the gain and the gain ratio of splitting
   by `id` into 14 groups. Compare with debt. Does the gain ratio solve the identifier trap?

2. **Categories on the 219.** Convert `ln_activos` into three categories (small, medium, large
   by terciles) and train `c50py` with that column as categorical and the other four as numeric.
   Which subset does it choose in the first split that uses size? Does the accuracy change?

3. **How much missing is too much.** Repeat the comparison of Section 3 with 10%, 30% and 50% of
   values erased. From what point does discarding incomplete rows stop being viable? And the
   median?

4. **How much a pure leaf pays.** With `_add_errs`, tabulate the pessimistic errors of a pure leaf
   of $N = 1, 2, 3, 5, 10, 20$ cases for `cf` 0.25 and 0.10. From what size does a pure leaf
   "pay for itself" (its penalty is less than one observed error)? What does that say about which leaves
   survive pruning?

5. **Rule per firm.** With `predict_rule`, get the rule applied to each test firm and
   count how many firms fall under each rule and what fraction of them is in default. Which rule is
   the "cleanest" and which the most doubtful?



## Solutions


In [ ]:

# 1. The identifier
cred_id = credito.copy(); cred_id["id"] = range(1, 15)
g, s, r = ganancia_y_ratio(cred_id, [cred_id["id"] == v for v in cred_id["id"]])
print(f"id: gain {g:.3f} (the maximum possible, H(paga) = {H(credito['paga']):.3f}), entropy of the partition {s:.3f}, gain ratio {r:.3f}")
g2, s2, r2 = ganancia_y_ratio(credito, [credito['deuda'].isin(['baja']), ~credito['deuda'].isin(['baja'])])
print(f"deuda {{baja}} against the rest: gain ratio {r2:.3f}")
print("The gain ratio penalizes id: from 0.985 to 0.259. In Notebook 1 that was not enough, because it was compared with debt split in three (0.185);")
print("with the binary split {baja} against the rest (0.338), debt beats id and the trap is resolved on this table. It is not a general guarantee:")
print("with fewer rows per value the entropy of the id partition grows less. That is why C4.5 adds another rule: only consider attributes with above-average gain.")


In [ ]:

# 2. ln_activos in three categories
terciles = X_train["ln_activos"].quantile([1/3, 2/3]).values
def cat_tam(v): return "chica" if v <= terciles[0] else ("mediana" if v <= terciles[1] else "grande")
X2tr = X_train.copy(); X2te = X_test.copy()
X2tr["ln_activos"] = X2tr["ln_activos"].map(cat_tam); X2te["ln_activos"] = X2te["ln_activos"].map(cat_tam)
m2 = C5Classifier(min_samples_leaf=10, numeric_threshold_strategy="all", categorical_features=["ln_activos"])
m2.fit(X2tr.values.astype(object), y_train.values, feature_names=RATIOS)
m2.print_tree(feature_names=RATIOS)
print(f"\nTest accuracy: {accuracy_score(y_test, m2.predict(X2te.values.astype(object))):.3f}   (with numeric ln_activos: {c5_def.score(Xte, y_test):.3f})")
print("Size in three categories appears as 'ln_activos in {chica}' at the second level: C5.0 grouped mediana and grande. With less information (three")
print("categories instead of the number) accuracy drops; the binary split by subsets is useful when the variable already comes as a category, not as a replacement for a number.")


In [ ]:

# 3. Different fractions of missing values
filas = []
for frac in [0.10, 0.20, 0.30, 0.50]:
    rng = np.random.default_rng(4)
    Xm = Xtr.copy(); Xm[rng.random(Xm.shape) < frac] = np.nan
    Xtm = Xte.copy(); Xtm[rng.random(Xtm.shape) < frac] = np.nan
    mediana = np.nanmedian(Xm, axis=0); Xi, Xti = np.where(np.isnan(Xm), mediana, Xm), np.where(np.isnan(Xtm), mediana, Xtm)
    completas = ~np.isnan(Xm).any(axis=1)
    a_c5 = C5Classifier(min_samples_leaf=10).fit(Xm, y_train.values).score(Xtm, y_test.values)
    a_med = DecisionTreeClassifier(min_samples_leaf=10, random_state=0).fit(Xi, y_train).score(Xti, y_test)
    a_comp = DecisionTreeClassifier(min_samples_leaf=10, random_state=0).fit(Xm[completas], y_train.values[completas]).score(Xti, y_test) if completas.sum() >= 20 else np.nan
    filas.append((f"{frac:.0%}", int(completas.sum()), round(a_c5, 3), round(a_med, 3), round(a_comp, 3) if not np.isnan(a_comp) else "fewer than 20 rows"))
print(pd.DataFrame(filas, columns=["missing", "complete rows (of 146)", "c50py", "CART median", "CART complete rows"]).to_string(index=False))
print("\nDiscarding rows stops being viable when only a few dozen remain (30% already leaves 26 rows). The median holds up longer, but with 50% gaps")
print("half of every column is an invented number and fractional propagation, which preserves the uncertainty, becomes the most honest option.")


In [ ]:

# 4. How much a pure leaf pays depending on its size
tabla4 = pd.DataFrame({cf: [round(_add_errs(N, 0, cf), 2) for N in [1, 2, 3, 5, 10, 20]] for cf in [0.25, 0.10]}, index=[1, 2, 3, 5, 10, 20])
tabla4.index.name = "N of the pure leaf"; tabla4.columns = ["cf = 0.25", "cf = 0.10"]
print(tabla4)
print("\nWith cf = 0.25 a pure leaf of 3 cases already pays 1.11 errors, more than one observed error; with 5, 1.21; with 20, 1.34: the penalty grows slowly")
print("(it tends to -ln(cf) = 1.39). So a small pure leaf only survives if, when merged with its sibling, the resulting node would make more than one")
print("extra error: pruning removes the leaves that separate one or two observations and keeps those that separate groups.")


In [ ]:

# 5. Rules per test firm
reglas = pd.Series(c5_def.predict_rule(Xte), index=X_test.index)
resumen = pd.DataFrame({"regla": reglas, "impago": y_test}).groupby("regla")["impago"].agg(["count", "mean"]).rename(columns={"count": "test firms", "mean": "fraction in default"}).round(2)
print(resumen.sort_values("test firms", ascending=False).to_string())
print("\nOn test, the cleanest rules are 'roa > 0.088 (with sales/debt > 1.278 and debt/assets > 0.586) => default' (4 of 4) and the big rule")
print("'sales/debt > 1.278 and debt/assets <= 0.586 => pays' (42 firms, 14% in default). The most doubtful is 'roa <= 0.088 => pays': of its 11 test")
print("firms, 7 fell into default; on training it was 20 against 12 and the majority decided 'pays'. It is the leaf a committee should look at first.")
